# EfficientNet-B6 — From Scratch in PyTorch
**Paper:** EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks (Tan & Le, ICML 2019)

| Config | Value |
|--------|-------|
| Width Coefficient  | 1.8 |
| Depth Coefficient  | 2.6 |
| Input Resolution   | 528×528 |
| Dropout Rate       | 0.5 |
| Parameters         | ~43M |

In [ ]:
# Cell 1 — Imports
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
# Cell 2 — Scaling Helpers
def make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def round_filters(filters, width_coefficient):
    return make_divisible(int(filters * width_coefficient))

def round_repeats(repeats, depth_coefficient):
    return int(math.ceil(repeats * depth_coefficient))

In [ ]:
# Cell 3 — Squeeze-and-Excitation Block
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.fc1     = nn.Conv2d(in_channels, reduced_dim, kernel_size=1)
        self.act     = nn.SiLU()
        self.fc2     = nn.Conv2d(reduced_dim, in_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        scale = self.pool(x)
        scale = self.act(self.fc1(scale))
        scale = self.sigmoid(self.fc2(scale))
        return x * scale

In [ ]:
# Cell 4 — MBConv Block (Mobile Inverted Bottleneck + SE)
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride,
                 expand_ratio, se_ratio=0.25, drop_path_rate=0.0):
        super().__init__()
        self.use_residual   = (in_channels == out_channels and stride == 1)
        self.drop_path_rate = drop_path_rate

        hidden_dim  = in_channels * expand_ratio
        reduced_dim = max(1, int(in_channels * se_ratio))

        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(),
            ]
        layers += [
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size,
                      stride=stride, padding=kernel_size // 2,
                      groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(),
            SqueezeExcitation(hidden_dim, reduced_dim),
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv(x)
        if self.use_residual:
            if self.drop_path_rate > 0 and self.training:
                keep = 1 - self.drop_path_rate
                mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
                return x + out * mask.float() / keep
            return x + out
        return out

In [ ]:
# Cell 5 — B0 Baseline Block Configuration
# (expand_ratio, out_channels, num_repeats, stride, kernel_size)
BASE_CONFIG = [
    (1,  16, 1, 1, 3),
    (6,  24, 2, 2, 3),
    (6,  40, 2, 2, 5),
    (6,  80, 3, 2, 3),
    (6, 112, 3, 1, 5),
    (6, 192, 4, 2, 5),
    (6, 320, 1, 1, 3),
]

In [ ]:
# Cell 6 — EfficientNet Model
class EfficientNet(nn.Module):
    def __init__(self, width_coefficient=1.0, depth_coefficient=1.0,
                 dropout_rate=0.2, num_classes=1000):
        super().__init__()

        in_ch = round_filters(32, width_coefficient)
        self.stem = nn.Sequential(
            nn.Conv2d(3, in_ch, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.SiLU(),
        )

        total_blocks = sum(
            round_repeats(rep, depth_coefficient)
            for _, _, rep, _, _ in BASE_CONFIG
        )
        block_idx = 0
        blocks    = []
        for expand, out_ch, reps, stride, ks in BASE_CONFIG:
            out_c    = round_filters(out_ch, width_coefficient)
            num_reps = round_repeats(reps,   depth_coefficient)
            for i in range(num_reps):
                dr = 0.2 * block_idx / total_blocks
                blocks.append(MBConv(
                    in_channels    = in_ch,
                    out_channels   = out_c,
                    kernel_size    = ks,
                    stride         = stride if i == 0 else 1,
                    expand_ratio   = expand,
                    drop_path_rate = dr,
                ))
                in_ch      = out_c
                block_idx += 1
        self.blocks = nn.Sequential(*blocks)

        head_ch = round_filters(1280, width_coefficient)
        self.head = nn.Sequential(
            nn.Conv2d(in_ch, head_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(head_ch),
            nn.SiLU(),
        )
        self.avgpool    = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(head_ch, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [ ]:
# Cell 7 — Factory Function & Instantiate EfficientNet-B6
def efficientnet_b6(num_classes=1000):
    return EfficientNet(width_coefficient=1.8, depth_coefficient=2.6,
                        dropout_rate=0.5, num_classes=num_classes)

NUM_CLASSES = 10
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

model = efficientnet_b6(num_classes=NUM_CLASSES).to(DEVICE)
print(model)

In [ ]:
# Cell 8 — Forward Pass Test
dummy  = torch.randn(2, 3, 528, 528).to(DEVICE)
output = model(dummy)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
# Cell 9 — Count Parameters
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'='*48}")
print(f"  Model           : EfficientNet-B6")
print(f"  Total params    : {total:,}")
print(f"  Trainable       : {trainable:,}")
print(f"  Non-trainable   : {total - trainable:,}")
print(f"{'='*48}")

In [ ]:
# Cell 10 — Layer-wise Parameter Breakdown
print(f"{'Layer':<50} {'Shape':<25} {'Params':>10}")
print('-' * 87)
for name, param in model.named_parameters():
    print(f"{name:<50} {str(list(param.shape)):<25} {param.numel():>10,}")
print('-' * 87)
print(f"{'TOTAL':<76} {total:>10,}")

---
## Training

In [ ]:
# Cell 11 — Dataset & DataLoader
DATA_DIR   = './data'
BATCH_SIZE = 8

train_transform = transforms.Compose([
    transforms.Resize(560),
    transforms.RandomCrop(528),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(560),
    transforms.CenterCrop(528),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

CLASS_NAMES = train_dataset.classes
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size   : {len(val_dataset)}')

In [ ]:
# Cell 12 — Training Loop
EPOCHS = 30
LR     = 5e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f"{'Epoch':<8} {'Tr Loss':<10} {'Tr Acc':<10} {'Val Loss':<10} {'Val Acc':<10}")
print('-' * 50)
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_acc:<10.2f} {vl_loss:<10.4f} {vl_acc:<10.2f}')

torch.save(model.state_dict(), 'efficientnet_b6_best.pth')
print('Model saved: efficientnet_b6_best.pth')

---
## Training Curves

In [ ]:
# Cell 13 — Training Curves (Loss & Accuracy)
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EfficientNet-B6 — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

---
## Inference

In [ ]:
# Cell 14 — Inference on a Single Image
from PIL import Image

def predict_image(image_path, top_k=5):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize(560),
        transforms.CenterCrop(528),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)

    top_probs, top_idx = torch.topk(probs, top_k, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_idx   = top_idx[0].cpu().tolist()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input Image')
    labels = [CLASS_NAMES[i] for i in top_idx]
    colors = ['#F44336' if i == 0 else '#2196F3' for i in range(top_k)]
    bars   = axes[1].barh(labels[::-1], [p*100 for p in top_probs[::-1]], color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    for bar, prob in zip(bars, top_probs[::-1]):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{prob*100:.1f}%', va='center')
    plt.tight_layout()
    plt.savefig('inference_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({top_probs[0]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
# Cell 15 — Collect Validation Predictions
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        probs = F.softmax(model(images.to(DEVICE)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f'Predictions shape : {all_probs.shape}')
print(f'Labels shape      : {all_labels.shape}')

In [ ]:
# Cell 16 — ROC AUC Curve (One-vs-Rest, per class)
y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')

macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'EfficientNet-B6 — ROC AUC Curve (Macro AUC = {macro_auc:.3f})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Macro AUC : {macro_auc:.4f}')
print('Per-class AUC:')
for cls, score in roc_auc_scores.items():
    print(f'  {cls:<20} {score:.4f}')